In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [9]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [ ]:
!pip uninstall -y comet
!pip install -q datasets==3.6.0 lightning transformers sentencepiece accelerate soundfile evaluate
!pip install -q unbabel-comet

In [ ]:
! pip install essentia==2.1b6.dev1389

In [ ]:
!pip uninstall -y tensorflow tensorflow-text tf-keras keras tensorboard

In [ ]:
from comet import download_model, load_from_checkpoint

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import lightning as L
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

MODEL_NAME = "openai/whisper-small"
SRC_LANG = "en_us"
TGT_LANG = "uk_ua"

BATCH_SIZE = 2
NUM_WORKERS = 0
MAX_EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-2
MAX_LABEL_LENGTH = 256
GENERATION_MAX_NEW_TOKENS = 128

OUTPUT_DIR = "ast_fleurs_outputs"
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().split())


def build_aligned_pairs(src_ds, tgt_ds) -> List[Dict[str, str]]:
    src_ids = {str(x["id"]) for x in src_ds}
    tgt_ids = {str(x["id"]) for x in tgt_ds}
    common_ids = sorted(src_ids & tgt_ids)

    tgt_text_by_id = {
        str(x["id"]): normalize_text(x["transcription"])
        for x in tgt_ds
        if str(x["id"]) in common_ids
    }

    pairs = [{"id": ex_id, "tgt_text": tgt_text_by_id[ex_id]} for ex_id in common_ids]
    return pairs


class FleursASTDataset(Dataset):
    def __init__(
        self,
        src_ds,
        aligned_pairs: List[Dict[str, str]],
        processor: WhisperProcessor,
        max_label_length: int = 256,
    ) -> None:
        self.src_ds = src_ds
        self.aligned_pairs = aligned_pairs
        self.processor = processor
        self.max_label_length = max_label_length

        self.src_idx_by_id = {str(x["id"]): i for i, x in enumerate(src_ds)}

    def __len__(self) -> int:
        return len(self.aligned_pairs)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        pair = self.aligned_pairs[idx]
        ex_id = pair["id"]

        src_item = self.src_ds[self.src_idx_by_id[ex_id]]

        audio = src_item["audio"]["array"]
        sr = src_item["audio"]["sampling_rate"]
        src_text = normalize_text(src_item["transcription"])
        tgt_text = pair["tgt_text"]

        input_features = self.processor.feature_extractor(
            audio,
            sampling_rate=sr,
            return_tensors="pt",
        ).input_features[0]

        labels = self.processor.tokenizer(
            tgt_text,
            truncation=True,
            max_length=self.max_label_length,
            return_tensors="pt",
        ).input_ids[0]

        return {
            "input_features": input_features,
            "labels": labels,
            "src_text": src_text,
            "tgt_text": tgt_text,
        }


@dataclass
class WhisperDataCollator:
    processor: WhisperProcessor

    def __post_init__(self):
        self.pad_token_id = self.processor.tokenizer.pad_token_id

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = torch.stack([f["input_features"] for f in features])

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"]
        labels = labels.masked_fill(labels == self.pad_token_id, -100)

        return {
            "input_features": input_features,
            "labels": labels,
            "src_texts": [f["src_text"] for f in features],
            "tgt_texts": [f["tgt_text"] for f in features],
        }


class WhisperASTLightningModule(L.LightningModule):
    def __init__(
        self,
        model_name: str = MODEL_NAME,
        lr: float = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        generation_max_new_tokens: int = GENERATION_MAX_NEW_TOKENS,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)
        self.processor = WhisperProcessor.from_pretrained(
            model_name,
            language="ukrainian",
            task="transcribe",
        )

        self.lr = lr
        self.weight_decay = weight_decay
        self.generation_max_new_tokens = generation_max_new_tokens

        self.model.generation_config.max_length = None

    def on_train_start(self):
        self.model.train()

    def forward(self, input_features: torch.Tensor, labels: torch.Tensor | None = None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch: Dict[str, Any], batch_idx: int):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        loss = outputs.loss
        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=batch["input_features"].size(0),
        )
        return loss

    def validation_step(self, batch: Dict[str, Any], batch_idx: int):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        val_loss = outputs.loss

        self.log(
            "val_loss",
            val_loss,
            prog_bar=True,
            on_step=False,
            on_epoch=True,
            batch_size=batch["input_features"].size(0),
        )

        generated_ids = self.model.generate(
            input_features=batch["input_features"],
            max_new_tokens=self.generation_max_new_tokens,
            language="uk",
            task="transcribe",
        )

        preds = self.processor.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        refs = self.processor.tokenizer.batch_decode(
            labels,
            skip_special_tokens=True,
        )

        return {
            "preds": preds,
            "refs": refs,
            "src_texts": batch["src_texts"],
        }

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


print("Loading FLEURS...")

en_train = load_dataset("google/fleurs", SRC_LANG, split="train", trust_remote_code=True)
uk_train = load_dataset("google/fleurs", TGT_LANG, split="train", trust_remote_code=True)

en_val = load_dataset("google/fleurs", SRC_LANG, split="validation", trust_remote_code=True)
uk_val = load_dataset("google/fleurs", TGT_LANG, split="validation", trust_remote_code=True)

en_test = load_dataset("google/fleurs", SRC_LANG, split="test", trust_remote_code=True)
uk_test = load_dataset("google/fleurs", TGT_LANG, split="test", trust_remote_code=True)

print("Building aligned pairs...")
train_pairs = build_aligned_pairs(en_train, uk_train)
val_pairs = build_aligned_pairs(en_val, uk_val)
test_pairs = build_aligned_pairs(en_test, uk_test)

print(f"Train size: {len(train_pairs)}")
print(f"Validation size: {len(val_pairs)}")
print(f"Test size: {len(test_pairs)}")

print("\nSample aligned pair:")
sample_id = train_pairs[0]["id"]
sample_src = en_train[[str(x["id"]) for x in en_train].index(sample_id)]["transcription"]
sample_tgt = train_pairs[0]["tgt_text"]
print("SRC:", sample_src)
print("TGT:", sample_tgt)


processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="ukrainian",
    task="transcribe",
)

train_ds = FleursASTDataset(
    src_ds=en_train,
    aligned_pairs=train_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

val_ds = FleursASTDataset(
    src_ds=en_val,
    aligned_pairs=val_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

test_ds = FleursASTDataset(
    src_ds=en_test,
    aligned_pairs=test_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

collator = WhisperDataCollator(processor)
use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

batch = next(iter(train_loader))
print("\nBatch shapes:")
print("input_features:", batch["input_features"].shape)
print("labels:", batch["labels"].shape)


model = WhisperASTLightningModule(
    model_name=MODEL_NAME,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    generation_max_new_tokens=GENERATION_MAX_NEW_TOKENS,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="whisper-ast-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

logger = CSVLogger(LOG_DIR, name="whisper_ast_fleurs")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if use_gpu else "cpu",
    devices=1,
    precision="16-mixed" if use_gpu else "32",
    logger=logger,
    callbacks=[
        checkpoint_callback,
        LearningRateMonitor(logging_interval="step"),
    ],
    log_every_n_steps=10,
)

trainer.fit(model, train_loader, val_loader)

print("\nBest checkpoint:")
print(checkpoint_callback.best_model_path)

Loading FLEURS...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Building aligned pairs...
Train size: 1425
Validation size: 138
Test size: 329

Sample aligned pair:
SRC: on monday scientists from the stanford university school of medicine announced the invention of a new diagnostic tool that can sort cells by type a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one u.s. cent each
TGT: у понеділок науковці зі школи медицини стенфордського університету оголосили про винайдення нового діагностичного інструменту що може сортувати клітини за їх видами це малесенький друкований чіп який можна виготовити за допомогою стандартних променевих принтерів десь по одному центу сша за штуку


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 350 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.



Best checkpoint:
/content/ast_fleurs_outputs/checkpoints/whisper-ast-epoch=01-val_loss=2.4099.ckpt


In [ ]:
import os
import torch
from tqdm import tqdm

best_ckpt = checkpoint_callback.best_model_path
print("Loading best checkpoint:", best_ckpt)

best_model = WhisperASTLightningModule.load_from_checkpoint(best_ckpt)
best_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
best_model.to(device)

all_src = []
all_ref = []
all_hyp = []

for batch in tqdm(test_loader, desc="Inference"):
    input_features = batch["input_features"].to(device)

    with torch.no_grad():
        generated_ids = best_model.model.generate(
            input_features=input_features,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            language="uk",
            task="transcribe",
        )

    preds = best_model.processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    all_src.extend(batch["src_texts"])
    all_ref.extend(batch["tgt_texts"])
    all_hyp.extend(preds)

print("Num predictions:", len(all_hyp))
print("\nExample prediction:")
print("SRC:", all_src[0])
print("REF:", all_ref[0])
print("HYP:", all_hyp[0])

src_path = os.path.join(PRED_DIR, "src.txt")
ref_path = os.path.join(PRED_DIR, "ref.txt")
hyp_path = os.path.join(PRED_DIR, "hyp.txt")

with open(src_path, "w", encoding="utf-8") as f:
    for x in all_src:
        f.write(x.strip() + "\n")

with open(ref_path, "w", encoding="utf-8") as f:
    for x in all_ref:
        f.write(x.strip() + "\n")

with open(hyp_path, "w", encoding="utf-8") as f:
    for x in all_hyp:
        f.write(x.strip() + "\n")

print("\nSaved:")
print(src_path)
print(ref_path)
print(hyp_path)

Loading best checkpoint: /content/ast_fleurs_outputs/checkpoints/whisper-ast-epoch=01-val_loss=2.4099.ckpt


Inference: 100%|██████████| 165/165 [05:36<00:00,  2.04s/it]

Num predictions: 329

Example prediction:
SRC: he did not set a figure for the cuts saying they will be made based on china's economic output
REF: він не встановив показник скорочень сказавши що це буде залежати від економічного обсягу виробництва китаю
HYP:  він не розказувався зі створенням що вони будуть створені зі дійсною економікою впливу війни на китайський інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаційний інформаці

Saved:
ast_fleurs_outputs/predictions/src.txt
ast_fleurs_outputs/predictions/ref.txt
ast_fleurs_outputs/predictions/hyp.txt


In [ ]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [
    {"src": s, "mt": h, "ref": r}
    for s, h, r in zip(all_src, all_hyp, all_ref)
]

comet_output = comet_model.predict(
    data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
)

print("COMET mean score:", comet_output.system_score)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads an

COMET mean score: 0.3593853818971698


In [2]:
# translate

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import lightning as L
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger


MODEL_NAME = "openai/whisper-small"
SRC_LANG = "en_us"
TGT_LANG = "uk_ua"

BATCH_SIZE = 2
NUM_WORKERS = 0
MAX_EPOCHS = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-2
MAX_LABEL_LENGTH = 256
GENERATION_MAX_NEW_TOKENS = 128

OUTPUT_DIR = "ast_fleurs_outputs"
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().split())


def build_aligned_pairs(src_ds, tgt_ds) -> List[Dict[str, str]]:
    src_ids = {str(x["id"]) for x in src_ds}
    tgt_ids = {str(x["id"]) for x in tgt_ds}
    common_ids = sorted(src_ids & tgt_ids)

    tgt_text_by_id = {
        str(x["id"]): normalize_text(x["transcription"])
        for x in tgt_ds
        if str(x["id"]) in common_ids
    }

    pairs = [{"id": ex_id, "tgt_text": tgt_text_by_id[ex_id]} for ex_id in common_ids]
    return pairs


class FleursASTDataset(Dataset):
    def __init__(
        self,
        src_ds,
        aligned_pairs: List[Dict[str, str]],
        processor: WhisperProcessor,
        max_label_length: int = 256,
    ) -> None:
        self.src_ds = src_ds
        self.aligned_pairs = aligned_pairs
        self.processor = processor
        self.max_label_length = max_label_length

        self.src_idx_by_id = {str(x["id"]): i for i, x in enumerate(src_ds)}

    def __len__(self) -> int:
        return len(self.aligned_pairs)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        pair = self.aligned_pairs[idx]
        ex_id = pair["id"]

        src_item = self.src_ds[self.src_idx_by_id[ex_id]]

        audio = src_item["audio"]["array"]
        sr = src_item["audio"]["sampling_rate"]
        src_text = normalize_text(src_item["transcription"])
        tgt_text = pair["tgt_text"]

        input_features = self.processor.feature_extractor(
            audio,
            sampling_rate=sr,
            return_tensors="pt",
        ).input_features[0]

        labels = self.processor.tokenizer(
            tgt_text,
            truncation=True,
            max_length=self.max_label_length,
            return_tensors="pt",
        ).input_ids[0]

        return {
            "input_features": input_features,
            "labels": labels,
            "src_text": src_text,
            "tgt_text": tgt_text,
        }


@dataclass
class WhisperDataCollator:
    processor: WhisperProcessor

    def __post_init__(self):
        self.pad_token_id = self.processor.tokenizer.pad_token_id

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = torch.stack([f["input_features"] for f in features])

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"]
        labels = labels.masked_fill(labels == self.pad_token_id, -100)

        return {
            "input_features": input_features,
            "labels": labels,
            "src_texts": [f["src_text"] for f in features],
            "tgt_texts": [f["tgt_text"] for f in features],
        }


class WhisperASTLightningModule(L.LightningModule):
    def __init__(
        self,
        model_name: str = MODEL_NAME,
        lr: float = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        generation_max_new_tokens: int = GENERATION_MAX_NEW_TOKENS,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)
        self.processor = WhisperProcessor.from_pretrained(
            model_name,
            language="ukrainian",
            task="translate",
        )

        self.lr = lr
        self.weight_decay = weight_decay
        self.generation_max_new_tokens = generation_max_new_tokens

        self.model.generation_config.max_length = None

    def on_train_start(self):
        self.model.train()

    def forward(self, input_features: torch.Tensor, labels: torch.Tensor | None = None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch: Dict[str, Any], batch_idx: int):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        loss = outputs.loss
        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=batch["input_features"].size(0),
        )
        return loss

    def validation_step(self, batch: Dict[str, Any], batch_idx: int):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        val_loss = outputs.loss

        self.log(
            "val_loss",
            val_loss,
            prog_bar=True,
            on_step=False,
            on_epoch=True,
            batch_size=batch["input_features"].size(0),
        )

        generated_ids = self.model.generate(
            input_features=batch["input_features"],
            max_new_tokens=self.generation_max_new_tokens,
            language="uk",
            task="translate",
        )

        preds = self.processor.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        refs = self.processor.tokenizer.batch_decode(
            labels,
            skip_special_tokens=True,
        )

        return {
            "preds": preds,
            "refs": refs,
            "src_texts": batch["src_texts"],
        }

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


print("Loading FLEURS...")

en_train = load_dataset("google/fleurs", SRC_LANG, split="train", trust_remote_code=True)
uk_train = load_dataset("google/fleurs", TGT_LANG, split="train", trust_remote_code=True)

en_val = load_dataset("google/fleurs", SRC_LANG, split="validation", trust_remote_code=True)
uk_val = load_dataset("google/fleurs", TGT_LANG, split="validation", trust_remote_code=True)

en_test = load_dataset("google/fleurs", SRC_LANG, split="test", trust_remote_code=True)
uk_test = load_dataset("google/fleurs", TGT_LANG, split="test", trust_remote_code=True)

print("Building aligned pairs...")
train_pairs = build_aligned_pairs(en_train, uk_train)
val_pairs = build_aligned_pairs(en_val, uk_val)
test_pairs = build_aligned_pairs(en_test, uk_test)

print(f"Train size: {len(train_pairs)}")
print(f"Validation size: {len(val_pairs)}")
print(f"Test size: {len(test_pairs)}")

print("\nSample aligned pair:")
sample_id = train_pairs[0]["id"]
sample_src = en_train[[str(x["id"]) for x in en_train].index(sample_id)]["transcription"]
sample_tgt = train_pairs[0]["tgt_text"]
print("SRC:", sample_src)
print("TGT:", sample_tgt)


processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="ukrainian",
    task="translate",
)

train_ds = FleursASTDataset(
    src_ds=en_train,
    aligned_pairs=train_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

val_ds = FleursASTDataset(
    src_ds=en_val,
    aligned_pairs=val_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

test_ds = FleursASTDataset(
    src_ds=en_test,
    aligned_pairs=test_pairs,
    processor=processor,
    max_label_length=MAX_LABEL_LENGTH,
)

collator = WhisperDataCollator(processor)
use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    pin_memory=use_gpu,
)

batch = next(iter(train_loader))
print("\nBatch shapes:")
print("input_features:", batch["input_features"].shape)
print("labels:", batch["labels"].shape)


model = WhisperASTLightningModule(
    model_name=MODEL_NAME,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    generation_max_new_tokens=GENERATION_MAX_NEW_TOKENS,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="whisper-ast-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

logger = CSVLogger(LOG_DIR, name="whisper_ast_fleurs")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if use_gpu else "cpu",
    devices=1,
    precision="16-mixed" if use_gpu else "32",
    logger=logger,
    callbacks=[
        checkpoint_callback,
        LearningRateMonitor(logging_interval="step"),
    ],
    log_every_n_steps=10,
)

trainer.fit(model, train_loader, val_loader)

print("\nBest checkpoint:")
print(checkpoint_callback.best_model_path)

Loading FLEURS...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Building aligned pairs...
Train size: 1425
Validation size: 138
Test size: 329

Sample aligned pair:
SRC: on monday scientists from the stanford university school of medicine announced the invention of a new diagnostic tool that can sort cells by type a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one u.s. cent each
TGT: у понеділок науковці зі школи медицини стенфордського університету оголосили про винайдення нового діагностичного інструменту що може сортувати клітини за їх видами це малесенький друкований чіп який можна виготовити за допомогою стандартних променевих принтерів десь по одному центу сша за штуку


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Batch shapes:
input_features: torch.Size([2, 80, 3000])
labels: torch.Size([2, 91])


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 350 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO: `Trainer.fit` stopped: `max_epochs=8` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=8` reached.



Best checkpoint:
/content/ast_fleurs_outputs/checkpoints/whisper-ast-epoch=02-val_loss=2.3810.ckpt


In [3]:
import os
import torch
from tqdm import tqdm

best_ckpt = checkpoint_callback.best_model_path
print("Loading best checkpoint:", best_ckpt)

best_model = WhisperASTLightningModule.load_from_checkpoint(best_ckpt)
best_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
best_model.to(device)

all_src = []
all_ref = []
all_hyp = []

for batch in tqdm(test_loader, desc="Inference"):
    input_features = batch["input_features"].to(device)

    with torch.no_grad():
        generated_ids = best_model.model.generate(
            input_features=input_features,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            language="uk",
            task="translate",
        )

    preds = best_model.processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    all_src.extend(batch["src_texts"])
    all_ref.extend(batch["tgt_texts"])
    all_hyp.extend(preds)

print("Num predictions:", len(all_hyp))
print("\nExample prediction:")
print("SRC:", all_src[0])
print("REF:", all_ref[0])
print("HYP:", all_hyp[0])

src_path = os.path.join(PRED_DIR, "src.txt")
ref_path = os.path.join(PRED_DIR, "ref.txt")
hyp_path = os.path.join(PRED_DIR, "hyp.txt")

with open(src_path, "w", encoding="utf-8") as f:
    for x in all_src:
        f.write(x.strip() + "\n")

with open(ref_path, "w", encoding="utf-8") as f:
    for x in all_ref:
        f.write(x.strip() + "\n")

with open(hyp_path, "w", encoding="utf-8") as f:
    for x in all_hyp:
        f.write(x.strip() + "\n")

print("\nSaved:")
print(src_path)
print(ref_path)
print(hyp_path)

Loading best checkpoint: /content/ast_fleurs_outputs/checkpoints/whisper-ast-epoch=02-val_loss=2.3810.ckpt


Inference: 100%|██████████| 165/165 [05:35<00:00,  2.03s/it]

Num predictions: 329

Example prediction:
SRC: he did not set a figure for the cuts saying they will be made based on china's economic output
REF: він не встановив показник скорочень сказавши що це буде залежати від економічного обсягу виробництва китаю
HYP: він не встановив зручну фікуру на кордоні стверджувача що вони будуть створюватися в основі на китайської економічної випадки випадків економічної випадки випадків економічної випадки випадків економічної випадків економічної випадків економічної випадків економічної в

Saved:
ast_fleurs_outputs/predictions/src.txt
ast_fleurs_outputs/predictions/ref.txt
ast_fleurs_outputs/predictions/hyp.txt


In [4]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [
    {"src": s, "mt": h, "ref": r}
    for s, h, r in zip(all_src, all_hyp, all_ref)
]

comet_output = comet_model.predict(
    data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
)

print("COMET mean score:", comet_output.system_score)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

COMET mean score: 0.35809296310672645
